# Лабораторная 1 · Инвертированный индекс и BM25 своими руками

**Неделя 3 · занятие 2.** Опора — L3 «Классический ИП»: инвертированный индекс, TF-IDF, BM25.

Сегодня ты собираешь своими руками то, что на лекции было на доске, и сверяешь **свои** числа
с числами лекции — исполняемо, а не на глаз. Если они разойдутся, упадёт ячейка, а не твоя вера
в семинар.

| # | вопрос занятия | чем отвечаем |
|---|---|---|
| 1 | Что именно даёт инвертированный индекс, если сравнить с линейным сканом? | строим оба, замеряем, сравниваем **отношение**, а не секунды |
| 2 | Почему TF-IDF ранжирует хуже BM25 и что конкретно чинят `k1` и `b`? | считаем обе формулы на одной коллекции и разбираем разницу по компонентам |
| 3 | Как узнать, что твоя реализация верна, а не просто «выдаёт числа»? | сверяем с `data/l3-*.json` — тем же источником, что питает слайды |

**Данные.** 20 Newsgroups (`sklearn.datasets.fetch_20newsgroups`) — де-факто свободный корпус,
входит в поставку scikit-learn. Плюс игрушечная коллекция из трёх документов (`cat`/`dog`/`mouse`),
сконструированная нами: на ней BM25 считается на бумаге за минуту.
Ничего под некоммерческой лицензией здесь нет — курс платный.

**Среда.** Бесплатный Colab T4 через плагин VS Code. GPU сегодня **не нужен вообще**: весь
семинар идёт на CPU. Артефакты — в папку на Drive, их подхватит `lab-cascade` на неделе 7.

**Бюджет: ≈120 минут.**

**Артефакт на вынос.** `artifacts/bm25_index.json` — твой инвертированный индекс по 20NG плюс
`run.json` с конфигурацией прогона и всеми замерами. На неделе 7 он станет первой ступенью каскада.

**Как запускать.** Сверху вниз, не пропуская ячеек. Ячейка `preflight()` упадёт заранее, если
окружение не то — это дешевле, чем упасть на середине.

<details><summary>Зачем вообще писать BM25 руками, если есть <code>rank_bm25</code> и Elasticsearch</summary>

Затем, что почти все ошибки поиска в проде — это не «плохая формула», а расхождение между тем,
что ты думаешь, что считаешь, и тем, что считает библиотека. У `sklearn.TfidfVectorizer` по
умолчанию `smooth_idf=True`, `sublinear_tf=False` и `norm='l2'` — три решения, ни одно из которых
не написано на доске, и все три меняют числа. Пока ты не посчитал формулу сам, ты не можешь
сказать, где именно разошлось.

Второе: BM25 — это три строки кода и два параметра, у которых есть физический смысл. Если ты
понимаешь, что делает `b`, ты умеешь чинить целый класс жалоб «длинные документы вылезают
наверх». Если не понимаешь — ты будешь крутить `k1` наугад.
</details>

<details><summary>Что ещё ломается в Colab, и почему preflight стоит именно здесь</summary>

Список в `preflight()` не выдуман. Каждая строка — реальный класс падения, и все они
объединены одним свойством: **падают не там, где причина**.

**numpy 2.x против скомпилированных колёс.** Colab обновляет numpy сам. Пакеты, собранные
под numpy 1.x, при импорте ловят ошибку ABI, и текст ошибки говорит про `_ARRAY_API`, а не
про версии. Хуже: `!pip install numpy==1.26.4` в Colab **не помогает без перезапуска рантайма** —
старый numpy уже загружен в память, и новый лежит на диске неиспользованным. Это первое,
что надо проверять, и первое, о чём забывают.

**Сеть.** `fetch_20newsgroups` при первом вызове тянет архив. Если сети нет или прокси режет,
падение случится в той ячейке, где ты этого не ждёшь, — например, в середине части 2, если
ты решил догрузить другую категорию. Мы проверяем доступность заранее и говорим об этом
человеческим текстом.

**Пути.** `DATA_DIR` может не существовать, если Drive не смонтирован или ноутбук запущен
локально. Сверка с лекцией тогда упадёт с `FileNotFoundError` в части 2 — через двадцать минут
после старта. Проверить существование папки стоит миллисекунду.

**Чего preflight НЕ ловит.** Он не ловит нехватку памяти: она проявится только на реальном
объёме, и предсказать её проверкой нельзя. Не ловит смену версии данных под тем же именем.
Не ловит того, что ты забыл перезапустить рантайм после установки — он видит уже загруженный
numpy и честно на него жалуется, но исправить не может.

**Принцип, который отсюда стоит унести.** Проверка окружения — не гигиена и не бюрократия.
Это перенос момента падения из середины работы в начало. Стоимость падения в начале — одна
минута; стоимость того же падения на двадцатой минуте — двадцать минут плюс подозрение,
что сломан твой код, а не окружение. Список проверок растёт только одним способом: каждый раз,
когда что-то сломалось не там, где причина, в `preflight()` добавляется строка.
</details>

## Шаг 0 · Пины и preflight

Версии зафиксированы. Список проверок в `preflight()` — не абстрактная гигиена, а перечень
**прошлых падений**: каждая строка когда-то стоила кому-то половины занятия.

In [ ]:
# ПИНЫ
!pip install -q numpy==1.26.4 scikit-learn==1.4.2 matplotlib==3.8.4

import json, math, os, random, re, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def preflight():
    problems = []
    if np.__version__.split(".")[0] != "1":
        problems.append(
            "numpy 2.x: sklearn 1.4.2 собран под numpy 1.x, импорт упадёт на ABI. "
            "Перезапусти рантайм после !pip install -- Colab держит старый numpy в памяти.")
    try:
        fetch_20newsgroups(subset="train", categories=["sci.space"], download_if_missing=False)
    except Exception:
        problems.append(
            "20NG не в кэше: первый вызов тянет ~14 МБ по сети. В Colab это норма, "
            "но если сети нет -- дальше всё упадёт, и упадёт не здесь, а через 20 минут.")
    if not Path(DATA_DIR).exists():
        problems.append(
            f"нет папки {DATA_DIR}: сверка с числами лекции невозможна. "
            "Смонтируй Drive или положи data/l3-*.json рядом с ноутбуком.")
    for p in problems:
        print("!", p)
    print("preflight:", "ЧИСТО" if not problems else f"{len(problems)} проблем(ы) -- читай выше")
    return not problems

## Шаг 1 · Конфигурация

Один блок, из которого управляется всё. Никаких форм Colab: интерфейс — VS Code, формы в нём
не отрисуются. Меняешь масштаб — меняешь переменную здесь, а не двадцать чисел по тексту.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SMOKE = os.environ.get("SMOKE", "0") == "1"   # быстрый прогон: меньше документов
N_DOCS = 200 if SMOKE else 2000               # размер рабочего среза 20NG
CATEGORIES = ["sci.space", "rec.sport.hockey"]
K1, B = 1.5, 0.75                             # параметры BM25 с доски L3

DRIVE_DIR = "/content/drive/MyDrive/dls-2026"
ARTIFACTS = Path(os.environ.get("ARTIFACTS", "./artifacts"))
DATA_DIR = os.environ.get("DLS_DATA", "./data")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

RUN = {"seed": SEED, "smoke": SMOKE, "n_docs": N_DOCS, "k1": K1, "b": B,
       "categories": CATEGORIES}      # сюда копятся ВСЕ замеры, в конце уходит в JSON
print(json.dumps(RUN, ensure_ascii=False))
preflight()

**Что видно.** Конфигурация напечатана целиком — это не украшение: в конце занятия она уедет
в `run.json` рядом с числами, и через месяц ты сможешь сказать, при каком `N_DOCS` получен любой
результат отсюда. Ожидать надо `smoke: false` и `n_docs: 2000`; если стоит `true`, ты в быстром
режиме, и все абсолютные числа ниже будут меньше — сравнивать их с числами соседа нельзя.
Чего этот вывод НЕ показывает: доступности сети и Drive — за это отвечает `preflight()` строкой
ниже. Что делать: если preflight не сказал «ЧИСТО», чини сейчас. Дальше поломка всплывёт через
двадцать минут и будет выглядеть как ошибка в твоём коде.

## Шаг 2 · Три документа, которые можно посчитать на бумаге

Прежде чем трогать две тысячи документов, берём три. Это та же коллекция, что была на доске
в L3, и её главное свойство — **ты можешь проверить машину руками**.

| док | текст | длина |
|---|---|---|
| D1 | `cat cat dog` | 3 |
| D2 | `cat dog dog mouse` | 4 |
| D3 | `mouse cat` | 2 |

Запрос: `cat dog`. Ожидаемый порядок с доски: **D2 > D1 > D3**.

In [ ]:
DOCS_TOY = {"D1": "cat cat dog", "D2": "cat dog dog mouse", "D3": "mouse cat"}
QUERY_TOY = ["cat", "dog"]

def build_index(docs):
    # docs: {docID: список токенов}. Тип один на весь ноутбук -- строку токенизируй заранее.
    postings = defaultdict(dict)
    lengths = {}
    for did, toks in docs.items():
        lengths[did] = len(toks)
        for term, tf in Counter(toks).items():
            postings[term][did] = tf
    return dict(postings), lengths

TOY = {d: t.split() for d, t in DOCS_TOY.items()}
post_toy, len_toy = build_index(TOY)
print("постинги:", json.dumps(post_toy, ensure_ascii=False))
print("длины:", len_toy, "· avgdl =", sum(len_toy.values()) / len(len_toy))

**Что видно.** Постинг-лист — это не «структура данных ради структуры»: у слова `cat` он длиной
три, у `dog` — два, у `mouse` — два. Сравнивать надо не термины друг с другом, а **длину списка
с числом документов**: `cat` встречается везде, значит его различающая сила нулевая, и любая
разумная формула обязана это учесть. Механизм ровно такой: `df` войдёт в `idf` со знаком минус.
Чего эта выдача НЕ показывает: порядка документов внутри списка — здесь он от `dict`, а в честном
поисковике постинги отсортированы по `docID`, иначе слияние двух списков перестаёт быть линейным.
Что делать: запомни `avgdl = 3.0` — это число войдёт в знаменатель BM25 и определит, кого
считать «длинным».

<details><summary>Почему постинг-лист обязан быть отсортирован, и что такое skip-pointer</summary>

Мы вернули постинги как `dict`, и порядок в нём — порядок вставки. Для игрушки это неважно,
для настоящего индекса — принципиально.

**Слияние двух отсортированных списков линейно.** Если оба постинг-листа отсортированы
по `docID`, пересечение считается одним проходом с двумя указателями: смотрим на головы обоих
списков, меньший двигаем вперёд, равные забираем в результат. Стоимость — сумма длин, `O(n + m)`.
Если списки не отсортированы, придётся строить хеш-множество из меньшего и проходить большим,
и это тоже `O(n + m)`, но с константой в разы больше и с памятью под множество. На запросе
из пяти терминов разница уже заметна.

**Skip-pointers.** Настоящая экономия начинается, когда списки сильно разной длины. Пусть
`space` встречается в миллионе документов, а `heliopause` — в двадцати. Проходить миллион
элементов ради двадцати совпадений глупо. В постинг-лист добавляют указатели «прыжка»: через
каждые `√n` элементов лежит ссылка на позицию дальше по списку. Алгоритм слияния, увидев,
что даже конец текущего прыжка меньше искомого `docID`, перескакивает целый блок. Получается
`O(√n)` вместо `O(n)` для длинного списка.

**Чего это НЕ даёт.** Skip-pointers помогают только конъюнктивным запросам (AND). При
дизъюнкции (OR) и при ранжировании по BM25 надо посмотреть каждый документ хотя бы одного
из списков — прыгать некуда. Именно поэтому в современных движках вместо прыжков применяют
WAND и block-max WAND: они отсекают документы, которые **заведомо не попадут в топ-k**,
по верхней границе вклада каждого термина. Это уже не про структуру списка, а про то, что
пользователю нужны десять результатов, а не все.

<summary>Как сделать правильно, если есть бюджет</summary>
Хранить постинги как отсортированный `array('i')` вместо `dict`, дописывать skip-pointers
через `int(len(postings) ** 0.5)` элементов и мерить не время слияния, а **число сравнений**:
время шумит, число сравнений — нет, и именно оно показывает, работает ли твоя оптимизация.
</details>

<details><summary>Как заставить sklearn считать формулу с доски — и стоит ли</summary>

Если очень хочется совпадения, его можно добиться, но цена поучительна.

**Три умолчания и что с ними делать.** `smooth_idf=True` добавляет единицу к `N` и к `df`
и ещё единицу к результату: `ln((1+N)/(1+df)) + 1`. Выключается флагом `smooth_idf=False`,
но и тогда остаётся `+1` в конце — он зашит и не отключается. `norm='l2'` нормирует вектор
документа на единичную длину; выключается `norm=None`, и тогда исчезает вся нормировка длины,
которой в TF-IDF и так нет по-человечески. `sublinear_tf=False` означает сырой `tf`;
включение даёт `1 + ln(tf)`, что ближе по духу к насыщению BM25, но это другая функция.

**Итог.** Даже с `smooth_idf=False, norm=None` совпадения с доской не будет из-за
неотключаемой `+1`. Полное совпадение достигается только своей реализацией — той, что мы
написали в двадцать строк.

**Стоит ли.** Обычно нет, и вот почему. `sklearn` не претендует на воспроизведение формулы
из учебника: он даёт разумную и численно устойчивую реализацию для задач машинного обучения,
где TF-IDF — это признаки для классификатора, а не ранжирующая функция. Сравнивать надо
не «кто правильнее», а «что ты делаешь». Если тебе нужны признаки — бери библиотеку. Если
тебе нужно ранжирование и ты будешь объяснять коллеге, почему документ на третьем месте, —
пиши формулу сам, иначе объяснение будет содержать фразу «ну там библиотека что-то делает».

**Общий урок, который дороже частного.** Всякий раз, когда твои числа расходятся с ожиданием,
есть два объяснения: ошибка у тебя или другое умолчание у библиотеки. Первое проверяется
за минуту сверкой с ручным расчётом на трёх элементах. Второе — чтением документации того
метода, который ты вызвал. Оба дешевле, чем полдня подозрений.
</details>

## Шаг 3 · Число, с которым сравнивается всё остальное

До первой «настоящей» формулы нужен тривиальный ранжировщик — самый тупой, какой можно
придумать. Иначе любая метрика ниже повиснет в вакууме: 0,62 — это хорошо или плохо?

Наш нулевой ранжировщик: **считать совпадения слов запроса, не взвешивая ничего.** Это
буквально «сколько раз встретилось». Всё, что мы построим дальше, обязано его обыгрывать —
а если не обыгрывает, это результат, а не повод спрятать таблицу.

In [ ]:
def rank_raw_count(docs, query):
    scores = {d: sum(toks.count(q) for q in query) for d, toks in docs.items()}
    return sorted(scores, key=lambda d: (-scores[d], d)), scores

order_base, scores_base = rank_raw_count(TOY, QUERY_TOY)
BASE = scores_base
ties = [s for s in set(scores_base.values()) if list(scores_base.values()).count(s) > 1]
print("BASE (сырой счёт):", scores_base)
print("порядок после тай-брейка по имени:", order_base)
print("порядок с доски:                  ['D2', 'D1', 'D3']")
print("НИЧЬИ на значениях:", ties or "нет")

**Что видно.** Главное здесь — не порядок, а строка про ничьи: у D1 и D2 сырой счёт **одинаков**,
по три совпадения у каждого. Сравнивать надо не выданный список с доской, а **значения между
собой**: список `D1, D2, D3` получился не потому, что D1 лучше, а потому что `sorted` разорвал
ничью по имени документа. Механизм очевиден: сырой счёт складывает вхождения, не различая,
какое слово встретилось, — два `cat` у D1 весят столько же, сколько `cat` плюс два `dog` у D2.
Чего этот результат НЕ показывает: что метод «почти работает». Он не работает вовсе — он просто
не различает эти два документа, а видимость порядка создаёт сортировка. Что делать: запомни свой
`BASE` как **словарь значений**, а не как список. Дальше каждая формула обязана объяснить,
за счёт чего она эту ничью разрывает осмысленно.

<details><summary>Посчитай BM25 для трёх документов на бумаге — полная арифметика</summary>

Это единственное место в курсе, где формулу можно проверить целиком карандашом. Дальше
коллекции станут слишком большими, и доверие придётся строить на `assert`, а не на глазах.

Дано: `D1 = cat cat dog` (длина 3), `D2 = cat dog dog mouse` (4), `D3 = mouse cat` (2).
Значит `N = 3`, `avgdl = (3 + 4 + 2) / 3 = 3,0`. Запрос: `cat dog`. Параметры `k1 = 1,5`, `b = 0,75`.

**Шаг 1, документные частоты.** `cat` есть во всех трёх документах, `df(cat) = 3`.
`dog` есть в D1 и D2, `df(dog) = 2`.

**Шаг 2, сглаженный idf.** Формула `ln((N - df + 0,5) / (df + 0,5) + 1)`.
Для `cat`: `ln((3 - 3 + 0,5) / (3 + 0,5) + 1) = ln(0,1429 + 1) = ln(1,1429) ≈ 0,1335`.
Для `dog`: `ln((3 - 2 + 0,5) / (2 + 0,5) + 1) = ln(0,6 + 1) = ln(1,6) ≈ 0,4700`.
Обрати внимание: `cat` есть везде, и его вес не ноль, а маленькое положительное число —
ровно то, ради чего в формуле стоит `+1`.

**Шаг 3, знаменатель нормировки.** `k1·(1 - b + b·dl/avgdl)` при `k1 = 1,5`, `b = 0,75`:
для D1 (`dl = 3 = avgdl`): `1,5·(0,25 + 0,75·1,0) = 1,5·1,0 = 1,5`;
для D2 (`dl = 4`): `1,5·(0,25 + 0,75·1,3333) = 1,5·1,25 = 1,875`;
для D3 (`dl = 2`): `1,5·(0,25 + 0,75·0,6667) = 1,5·0,75 = 1,125`.
Видно, что длинный документ получает больший знаменатель, то есть насыщается медленнее.

**Шаг 4, вклады.** Компонента `tf·(k1+1) / (tf + знаменатель)`, то есть `tf·2,5 / (tf + Z)`.
D1: `cat` при `tf = 2` даёт `2·2,5 / (2 + 1,5) = 5 / 3,5 ≈ 1,4286`, умножаем на `idf(cat)`:
`1,4286·0,1335 ≈ 0,1907`. `dog` при `tf = 1`: `2,5 / 2,5 = 1,0`, умножаем на `0,47` → `0,4700`.
Сумма D1 ≈ **0,6607**.
D2: `cat` при `tf = 1`: `2,5 / (1 + 1,875) = 0,8696`, ×`0,1335` ≈ `0,1161`. `dog` при `tf = 2`:
`5 / (2 + 1,875) = 1,2903`, ×`0,47` ≈ `0,6064`. Сумма D2 ≈ **0,7225**.
D3: `cat` при `tf = 1`: `2,5 / (1 + 1,125) = 1,1765`, ×`0,1335` ≈ `0,1571`. `dog` отсутствует,
вклад ноль. Сумма D3 ≈ **0,1571**.

**Итог: D2 > D1 > D3** — ровно порядок с доски. Обрати внимание, почему D2 обошёл D1: не
из-за длины и не из-за общего числа совпадений (у обоих их три), а из-за того, что лишнее
вхождение у D2 пришлось на **редкое** слово `dog`, а у D1 — на частое `cat`. Это и есть
работа `idf` в чистом виде, видимая на трёх документах.

**Чего этот расчёт НЕ показывает.** Он не показывает, что D2 действительно релевантнее для
человека. Формула ранжирует по своей модели важности; совпадение с интуицией на игрушке —
приятное свойство, а не доказательство.
</details>

⚠️ Ловушка C · **Порядок в выдаче может не нести информации вообще.** Если бы мы напечатали
только список, мы бы увидели правдоподобный ранжированный ответ и не узнали, что два первых
места разделены не оценкой, а алфавитом. Ничьи в поиске повсеместны — особенно на коротких
документах и булевых признаках, — и почти всегда невидимы: API возвращает список, а не баллы.
Правило: **печатай баллы, а не только порядок.** Иначе тай-брейк твоей библиотеки будет молча
выдавать себя за качество ранжирования.

⚠️ Ловушка D · **`sorted` стабилен, и это не спасает.** Питоновская сортировка сохраняет порядок
равных элементов — то есть результат зависит от порядка вставки в словарь, а он у нас от порядка
чтения файлов. Поменяется порядок чтения — поменяется выдача при неизменных баллах. Мы добавили
явный вторичный ключ `d` в `sorted(..., key=lambda d: (-scores[d], d))`, чтобы результат был
воспроизводим; без него ноутбук выдавал бы разное на разных машинах, не выдавая ни одной ошибки.

---

## Часть 1 · Инвертированный индекс — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 1.1 | Насколько линейный скан хуже индекса? | замеряем оба на 2000 документах |
| 1.2 | Что мы потеряли, построив индекс? | считаем цену построения и памяти |
| 1.3 | Совпадает ли наш индекс с индексом лекции? | сверяем `df` и постинги с `data/l3-index.json` |

### Шаг 1.1 · Загружаем корпус и смотрим на него **до** любых формул

Первый пункт чек-листа в конце занятия — «посмотреть на данные до выбора метода». Выполним его
прямо сейчас, а не в конце.

In [ ]:
raw = fetch_20newsgroups(subset="train", categories=CATEGORIES,
                         remove=("headers", "footers", "quotes"), random_state=SEED)
TOKEN = re.compile(r"[a-z]{2,}")

def tokenize(text):
    return TOKEN.findall(text.lower())

pairs = [(f"N{i}", tokenize(t)) for i, t in enumerate(raw.data)]
pairs = [(d, toks) for d, toks in pairs if len(toks) >= 5][:N_DOCS]
CORPUS = dict(pairs)
lens = np.array([len(t) for t in CORPUS.values()])

print(f"документов: {len(CORPUS)} · токенов всего: {lens.sum()}")
print(f"длина: медиана {np.median(lens):.0f}, среднее {lens.mean():.1f}, "
      f"максимум {lens.max()} (в {lens.max() / np.median(lens):.0f}x длиннее медианы)")
RUN["n_docs_real"], RUN["avgdl"] = len(CORPUS), float(lens.mean())

**Что видно.** Среднее заметно больше медианы — это и есть тот самый перекос, ради которого
в BM25 существует параметр `b`. Сравнивать надо не среднее с медианой абстрактно, а **максимум
с медианой**: самый длинный документ в десятки раз длиннее типичного. Механизм простой: в 20NG
попадаются простыни с цитатами и подписями, и любая формула, которая складывает частоты без
нормировки, отдаст им верх выдачи просто за объём. Чего эти числа НЕ показывают: содержания —
длинный документ вполне может быть релевантным, мы пока ничего об этом не знаем. Что делать:
запомнить `avgdl` и вернуться к нему в части 3, когда будем крутить `b`.

<details><summary>Почему медиана и среднее расходятся, и когда это ломает не только BM25</summary>

Расхождение медианы и среднего — это не свойство 20NG, а свойство почти любого корпуса
текстов, написанных людьми. Длины документов распределены примерно логнормально: короткие
встречаются часто, длинные — редко, но они очень длинные. У логнормального распределения
среднее всегда правее медианы, и тем сильнее, чем толще хвост.

**Что ломается.** Первое — `avgdl` в BM25: он считается средним, и один документ на сто тысяч
слов сдвигает его для всех остальных. Второе — любая оценка «типичного» размера: если ты
планируешь память под чанки по среднему, тебе не хватит. Третье — батчинг при кодировании
эмбеддингов: батч по числу документов даёт скачущее время, батч по числу токенов — ровное.
Это аукнется на семинаре недели 7.

**Чего это НЕ означает.** Не означает, что среднее «неправильное», а медиана «правильная».
В BM25 нужно именно среднее: формула нормирует на ожидаемую длину, а не на типичную, и это
осознанный выбор Робертсона. Не означает и того, что длинные документы плохи — в 20NG длинный
тред часто содержательнее короткой реплики.

**Проверка устойчивости, которую мы не сделали.** Честно было бы посчитать `avgdl` на
десяти бутстрап-подвыборках и посмотреть, насколько он гуляет. Если `avgdl` пляшет на
десятки процентов, все числа BM25 ниже пляшут вместе с ним, и сравнивать конфигурации,
посчитанные на разных подвыборках, нельзя вообще. Мы этого не сделали ради времени —
и это ровно тот случай, когда ограничение надо назвать вслух, а не спрятать.

<summary>Как сделать правильно, если есть бюджет</summary>
`np.percentile(lens, [50, 90, 99])` рядом со средним, и обрезка по 99-му перцентилю с явной
пометкой, сколько документов обрезано. Обрезать молча нельзя: это меняет `N`, а с ним `idf`.
</details>

⚠️ Ловушка A · **`remove=("headers", "footers", "quotes")` стоит не для красоты.** Без него
в тексте остаётся строка `Newsgroups: sci.space`, и любой классификатор или ретривер получает
ответ прямо во входе. Это классическая утечка 20NG: работы, которые её не убирают, показывают
подозрительно высокое качество и не понимают почему. Мы её убрали — и наши числа поэтому будут
**ниже** опубликованных на «сыром» 20NG. Это смещение в нашу пользу по честности и не в нашу
пользу по цифрам.

In [ ]:
plt.figure(figsize=(9, 3))
plt.hist(lens, bins=60, color="#3B6FD4")
plt.axvline(np.median(lens), color="#B4521F", label=f"медиана {np.median(lens):.0f}")
plt.axvline(lens.mean(), color="#111", ls="--", label=f"среднее {lens.mean():.0f}")
plt.xlabel("длина документа, токенов"); plt.ylabel("документов"); plt.legend()
plt.title("Распределение длин: хвост справа -- это будущая работа параметра b")
plt.tight_layout(); plt.show()

**Что видно.** Распределение с тяжёлым правым хвостом: масса документов слева, отдельные
простыни справа. Сравнивать надо не столбики друг с другом, а **положение двух вертикалей**:
среднее вытянуто хвостом вправо относительно медианы, и именно среднее (`avgdl`) стоит
в знаменателе BM25. Ожидаемая картина ровно такая по механизму — почтовые треды растут
цитированием, а не содержанием. Чего гистограмма НЕ показывает: связи длины с релевантностью,
и это отдельный вопрос, к которому мы вернёмся заданием 3. Что делать: не «нормировать на всякий
случай». Соблазн очевиден — «ну так поделим на длину» — и он хуже, чем кажется: полная нормировка
(`b=1`) убивает сигнал у длинных релевантных документов. `b` существует, чтобы нормировать
**частично**.

### Шаг 1.2 · Скан против индекса

Теперь замер, который закрывает спор из вопроса 1. Меряем **отношение**, а не секунды:
секунды на твоей машине ничего не говорят о секундах на моей.

In [ ]:
QUERY = ["space", "team"]

def scan_search(corpus, query):
    hits = []
    for did, toks in corpus.items():
        s = set(toks)
        if all(q in s for q in query):
            hits.append(did)
    return hits

post, doclen = build_index(CORPUS)

def index_search(postings, query):
    lists = [set(postings.get(q, {})) for q in query]
    return sorted(set.intersection(*lists)) if lists and all(lists) else []

t0 = time.perf_counter(); [scan_search(CORPUS, QUERY) for _ in range(5)]
t_scan = (time.perf_counter() - t0) / 5
t0 = time.perf_counter(); [index_search(post, QUERY) for _ in range(5)]
t_idx = (time.perf_counter() - t0) / 5

hits_scan, hits_idx = scan_search(CORPUS, QUERY), index_search(post, QUERY)
assert set(hits_scan) == set(hits_idx), "скан и индекс нашли РАЗНЫЕ документы"
assert hits_scan != hits_idx or len(hits_scan) < 2, \
    "ожидали РАЗНЫЙ порядок при одинаковом множестве -- см. ловушку про лексикографию ниже"
RUN["speedup"] = t_scan / t_idx
print(f"скан {t_scan * 1e3:.2f} мс · индекс {t_idx * 1e3:.3f} мс · отношение {t_scan / t_idx:.0f}x")
print(f"словарь: {len(post)} терминов · постингов всего: {sum(len(v) for v in post.values())}")

**Что видно.** Отношение — в сотни раз, и `assert` выше говорит важнейшее: оба метода нашли
**одно и то же множество**. Сравнивать надо именно отношение: абсолютные миллисекунды на
бесплатном Colab плавают втрое от соседа по гипервизору и не переносятся никуда. Механизм: скан
трогает каждый документ корпуса, индекс — только два постинг-листа. А теперь неприятное, и это
главное здесь: **такое отношение само по себе подозрительно.** Ниже, в блоке про то, чего этот
замер не показывает, сформулировано правило — ускорение больше чем в сто раз обычно означает,
что baseline реализован небрежно. Наш скан пересобирает `set(toks)` на каждом документе при
каждом запросе, то есть мы сравнили индекс с заведомо плохим сканом. Чего замер НЕ показывает:
цены построения индекса, которую мы заплатили заранее и не включили в таймер. Что делать:
прочитать нижний слой целиком и не цитировать это число никому — оно измеряет в том числе
качество нашего же baseline.

<details><summary>Чего этот замер НЕ показывает — полный список</summary>

Мы напечатали отношение и объявили победителя. Вот всё, что осталось за кадром, и почему
каждый пункт может перевернуть вывод.

1. **Цена построения.** Индекс строился примерно столько же, сколько идёт сотня сканов.
   Экономика индекса держится на том, что запросов много, а построение одно. При одном
   запросе на корпус скан выигрывает с разгромом.
2. **Память.** Постинги занимают порядок размера корпуса. Мы это увидим в финальной ячейке,
   где артефакт ляжет на диск. На корпусе, который не влезает в оперативную память, весь
   разговор про скорость меняется: начинается разговор про то, сколько раз ты пойдёшь на диск.
3. **Обновление.** Добавить документ в скан стоит ноль. Добавить в индекс — надо тронуть
   постинг-лист каждого термина документа. Настоящие движки решают это сегментами и слиянием
   в фоне, и вся сложность Lucene примерно про это.
4. **Тип запроса.** Мы мерили конъюнкцию двух частых терминов. На запросе из одного очень
   частого термина индекс вернёт почти весь корпус, и выигрыш схлопнется.
5. **Реализация скана.** Наш скан строит `set(toks)` на каждом документе при каждом запросе —
   это заведомо глупо. Честный скан кэшировал бы множества, и отношение упало бы в разы.
   Мы сравнили индекс с **плохим** сканом, и это смещение в нашу пользу.

Пятый пункт — самый важный и самый частый в чужих бенчмарках: baseline реализуют небрежно,
а свой метод вылизывают. Отсюда правило: если твоё ускорение больше, чем в сто раз, скорее
всего ты сравниваешься с чем-то сломанным.
</details>

⚠️ Ловушка E · **Замер без прогрева врёт.** Первый вызов `scan_search` тянет данные в кэш
процессора и платит за это. Мы усреднили по пяти прогонам — но не выкинули первый, и это
сознательное упрощение в пользу простоты кода. На честном бенчмарке первый прогон отбрасывают,
а разброс показывают явно. <details><summary>как сделать правильно, если есть бюджет</summary>
`timeit.repeat(..., repeat=7, number=10)`, брать минимум (не среднее: шум только добавляет
время, а не вычитает), и печатать все семь чисел, чтобы студент увидел разброс своими
глазами.</details>

⚠️ Ловушка D · **`sorted` по строковым id — это лексикографический порядок, не числовой.**
Наши документы называются `N0, N1, ..., N1145`, и `sorted` расставит их как `N0, N1, N10, N100,
N1000` — не как человек ожидает. Поэтому в ячейке выше мы сравниваем **множества**, а не списки:
скан отдаёт документы в порядке обхода корпуса, индекс — в лексикографическом, и оба правы.
Второй `assert` намеренно требует, чтобы порядки **различались**: если они вдруг совпадут,
значит id стали такими, что лексикография совпала с обходом, и ловушка перестала быть видимой.
Практическое следствие серьёзнее опрятности: слияние постинг-листов линейно только при
**согласованном** порядке. Смешать два списка, отсортированных по-разному, — значит получить
неверное пересечение, не получив ни одной ошибки. В настоящих движках docID — целое число
именно поэтому.

⚠️ Ловушка B · **`5x` быстрее — это про что?** Ускорение поиска по индексу не равно ускорению
поисковой системы. В проде между запросом и ответом лежат сеть, парсинг запроса, ранжирование,
сниппеты и рендер. Ускорив 2% времени в 100 раз, ты получишь 2% выигрыша, а не 100. Это тот же
класс ошибки, что «Recall@100 = 0,95» в системе, где пользователю показывают три ссылки.

### Шаг 1.3 · Сверка с лекцией

Здесь начинается то, чего почти не бывает в семинарах: **исполняемая** сверка с доской.
`data/l3-index.json` — тот же файл, из которого построены слайды L3. Если наш индекс
разойдётся с лекционным, упадёт ячейка.

In [ ]:
LEC = json.load(open(f"{DATA_DIR}/l3-index.json", encoding="utf-8"))
lec_docs = {d["id"]: d for d in LEC["docs"]}
toy8 = {d["id"]: d["len"] for d in LEC["docs"]}

print("лекция: N =", LEC["N"], "· запрос:", LEC["query"])
for term, info in LEC["terms"].items():
    print(f"  {term:6} df={info['df']}  постинги={info['postings']}")
print("наши df на том же запросе (по 20NG-срезу, НЕ те же 8 документов):",
      {q: len(post.get(q, {})) for q in LEC["query"]})

**Что видно.** Лекционные `df` равны четырём для обоих терминов, и это **не совпадает** с нашими
числами — так и должно быть. Сравнивать надо не наши `df` с лекционными, а **устройство** записи:
термин → список документов плюс `df`. Лекция взяла восемь самых коротких документов, чтобы
таблица влезла на слайд; мы взяли две тысячи. Механизм расхождения полностью объяснён размером
выборки. Чего эта распечатка НЕ показывает: правильности нашей формулы — до неё мы доберёмся
в части 2, где сверка станет побитовой. Что делать: не пытаться подогнать наши `df` под
лекционные. Расхождение объяснено — это разрешённый исход, а не ошибка.

<details><summary>Почему запрос именно «space team» и что было бы на другом</summary>

Запрос выбран не случайно и не потому, что красивый. У него три полезных свойства, и стоит
понимать каждое, потому что на другом запросе часть сегодняшних выводов изменится.

**Свойство первое: два термина примерно равной частоты.** Обе категории корпуса
(`sci.space` и `rec.sport.hockey`) представлены поровну, поэтому `space` и `team` имеют
сопоставимые `df`. Это делает наглядной работу `idf`: веса терминов близки, и разницу
в ранжировании создаёт частота внутри документа, а не разница весов. На запросе из очень
частого и очень редкого слова картина была бы противоположной — редкое слово задавило бы
всё, и `k1` с `b` почти перестали бы влиять.

**Свойство второе: термины разделяют корпус.** `space` тянет одну категорию, `team` — другую.
Конъюнкция даёт документы, где встречается и то и другое, — то есть либо про космический
спорт (таких нет), либо про что-то нейтральное, где оба слова случайны. Это делает
пересечение маленьким и стабильным, что удобно для замера скорости, но означает, что
наши топ-5 набраны в основном дизъюнкцией.

**Свойство третье: оба термина частые, значит постинг-листы длинные.** Именно поэтому выигрыш
индекса над сканом получился скромнее, чем мог бы: индекс всё равно проходит тысячи элементов.
На редком термине выигрыш был бы драматичнее, и это стоит проверить самому — заменить запрос
на что-то вроде `["heliopause"]` и посмотреть на отношение.

**Что отсюда следует для чтения чужих бенчмарков.** Набор запросов определяет выводы не меньше,
чем алгоритм. Работа, которая меряет ускорение индекса на редких терминах, и работа, которая
меряет его на частых, получат числа, отличающиеся на порядок, и обе будут правы. Первое, что
надо искать в чужом замере поиска, — как выбраны запросы. Если про это не написано, число
не значит ничего.
</details>

⚠️ Ловушка A · **Восемь самых коротких документов — искусственный масштаб.** В `data/l3-*.json`
лежит ровно та выборка, которая помещается на слайд. Все числа лекции про эти восемь документов
верны, и ни одно из них не переносится на корпус: `avgdl = 50,6` там и сотни здесь. Toy
воспроизводит **явление и порядок**, а не величину.

---

## Часть 2 · TF-IDF и почему он ломается — 30 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 2.1 | Что именно считает `idf`? | считаем руками и сверяем с `data/l3-bm25.json` побитово |
| 2.2 | Почему сырые частоты не работают? | строим контрпример и показываем срыв |
| 2.3 | Совпадёт ли `sklearn` с формулой с доски? | не совпадёт — и это главный урок шага |

In [ ]:
BM = json.load(open(f"{DATA_DIR}/l3-bm25.json", encoding="utf-8"))
N_LEC, AVGDL_LEC = BM["N"], BM["avgdl"]

def idf_classic(df, n):
    return math.log(n / df)

mismatch = []
for d in BM["docs"]:
    for t in d["terms"]:
        got = round(idf_classic(t["df"], N_LEC), 4)
        if abs(got - t["idf"]) > 1e-4:
            mismatch.append((d["id"], t["t"], got, t["idf"]))

assert not mismatch, f"idf разошёлся с лекцией: {mismatch[:3]}"
print(f"сверка idf с data/l3-bm25.json: {sum(len(d['terms']) for d in BM['docs'])} значений совпали")
print("формула лекции: idf = ln(N/df), N =", N_LEC)

**Что видно.** Все значения `idf` из лекции воспроизвелись формулой `ln(N/df)` — это и есть
проверка, что мы поняли доску правильно, а не «примерно так». Сравнивать надо не наши числа
с интуицией, а с файлом, из которого сделан слайд: расхождение стало бы падающей ячейкой,
а не незамеченным противоречием через три занятия. Механизм: `idf` наказывает частые термины
логарифмически, поэтому слово из половины корпуса весит примерно `0,69`, а из одной восьмой —
`2,08`. Чего эта сверка НЕ показывает: что классический `idf` вообще хорош — у него есть
неприятность, которую мы увидим через две ячейки. Что делать: держать в голове, что это
**одна из** формул `idf`, а не единственная.

<details><summary>Что ещё умеет молча испортить токенизация — и почему это важнее формулы</summary>

Мы токенизируем регуляркой `[a-z]{2,}` по нижнему регистру. Это решение принято в одну строку
и влияет на результат сильнее, чем выбор между TF-IDF и BM25.

**Что мы выбросили.** Однобуквенные токены: `a`, `I`, но заодно и `C` — язык программирования.
Цифры целиком: `Apollo 11` стал `apollo`, а `Ariane 5` и `Ariane 6` стали неразличимы. Дефисы:
`state-of-the-art` распался на четыре токена, из которых три бессмысленны. Апострофы:
`don't` → `don` + `t`, и `t` тоже выброшен как однобуквенный.

**Что мы не сделали.** Стемминг и лемматизацию. Портеровский стеммер свёл бы `running`,
`runs`, `ran` к общей основе, подняв полноту и уронив точность: `universe` и `university`
стеммер Портера отправляет в одну основу `univers`, и запрос про вселенную начнёт находить
университеты. Это классический пример, и он реален.

**Почему это важнее формулы.** Разница между TF-IDF и BM25 на типичной коллекции — единицы
процентов метрики. Разница между «со стеммингом» и «без» на морфологически богатом языке —
десятки процентов. На русском, где у существительного дюжина форм, лексический поиск без
нормализации форм просто не работает, и об этом целая лекция L19.

**Проверка устойчивости, которую стоит сделать.** Прогнать весь ноутбук с токенизатором
`[a-z0-9]{2,}` вместо `[a-z]{2,}` и посмотреть, изменился ли топ-5. Если изменился сильно —
твои выводы про `b` были про токенизатор, а не про `b`. Меняешь одно — меряешь это одно.

<summary>Как сделать правильно, если есть бюджет</summary>
Токенизатор выносится в переменную конфигурации рядом с `SEED`, и весь ноутбук прогоняется
дважды — с ним и с альтернативой. Разница между двумя прогонами и есть цена этого решения,
выраженная в числах, а не в рассуждениях.
</details>

<details><summary>Откуда в idf взялся логарифм — вывод, а не мнемоника</summary>

Логарифм в `idf` — не «чтобы сгладить», а следствие вероятностной постановки.

**Шаг первый.** Пусть мы хотим оценить, насколько термин `t` неожидан. Вероятность встретить
его в случайном документе оценим долей: `p(t) = df / N`. Информация, которую несёт наблюдение
события с вероятностью `p`, по Шеннону равна `-log p`. Подставляем: `-log(df/N) = log(N/df)`.
Это ровно классический `idf`. То есть `idf` — это **собственная информация** термина,
и лекция L4 про энтропию говорит о том же самом с другой стороны.

**Шаг второй, откуда сглаживание.** В вероятностной модели релевантности Робертсона-Спарк
Джонс вес термина выводится как логарифм отношения шансов: насколько чаще термин встречается
в релевантных документах, чем в нерелевантных. Без разметки релевантные оценивают как «мало»,
и формула вырождается в `log((N - df + 0,5) / (df + 0,5))`. Половинки — это сглаживание
Джеффриса, поправка на нулевые счётчики; без неё формула взрывается при `df = 0`.

**Шаг третий, откуда `+1`.** Отношение шансов уходит в минус, когда термин встречается больше
чем в половине документов. Для вероятностной модели это осмысленно: термин свидетельствует
**против** релевантности. Для практического поиска это катастрофа — документ штрафуется
за содержание слова из запроса. `+1` под логарифмом сдвигает всю кривую так, что она
остаётся положительной при любом `df`, сохраняя монотонность. Это инженерная поправка,
а не теорема, и в этом честно признаются сами авторы BM25.

**Чего это НЕ означает.** Не означает, что `idf` — «правильная» мера важности. Он не знает
ни о синонимии, ни о морфологии, ни о том, что слово может быть частым в корпусе и редким
в предметной области. Все три дырки чинятся в следующих лекциях, и ни одна — подкруткой `idf`.
</details>

In [ ]:
counts = {"частый (df=N)": idf_classic(N_LEC, N_LEC),
          "половина (df=N/2)": idf_classic(N_LEC / 2, N_LEC),
          "редкий (df=1)": idf_classic(1, N_LEC)}
for k, v in counts.items():
    print(f"{k:20} idf = {v:.4f}")
print("BM25 использует сглаженный вариант: ln((N-df+0.5)/(df+0.5) + 1)")
print("  для df=N он даёт:", round(math.log((N_LEC - N_LEC + 0.5) / (N_LEC + 0.5) + 1), 4))

**Что видно.** Классический `idf` для термина, который есть **во всех** документах, равен ровно
нулю: `ln(N/N) = 0`. Сравнивать надо две последние строки: сглаженный вариант BM25 в той же
ситуации даёт не ноль, а маленькое положительное число. Ожидаемая картина такая по механизму —
`+0,5` в числителе и знаменателе и `+1` под логарифмом не дают весу схлопнуться и, что важнее,
уйти в минус: у классической формулы Робертсона-Спарк Джонс без сглаживания вес частого термина
становится **отрицательным**, и документ штрафуется за содержание слова из запроса. Чего этот
расчёт НЕ показывает: насколько это важно на практике — на нашем корпусе термина с `df = N` нет.
Что делать: не удивляться, что в части 3 числа `idf` будут другими. Это не ошибка, это другая
формула, и BM25 всегда идёт со своей.

<details><summary>Почему отрицательный вес — это не абстрактная опасность</summary>

Легко решить, что случай `df > N/2` экзотический. Он не экзотический — он повседневный,
как только в запросе появляется служебное слово.

Возьми запрос «как настроить принтер». Слово «как» встречается почти в каждом документе.
При несглаженном RSJ его вес отрицателен, и документ, где написано «как настроить принтер»,
получает штраф относительно документа, где написано «настроить принтер». То есть **точное
совпадение с запросом проигрывает неточному** — ровно противоположно ожиданиям пользователя.

Исторически с этим боролись стоп-словами: слова с высоким `df` просто выкидывали из индекса.
Решение работает и имеет неприятную цену: запрос «to be or not to be» после удаления
стоп-слов становится пустым. Современные движки стоп-слова не удаляют, а полагаются на
сглаженный `idf` плюс на то, что вклад частого термина мал, но неотрицателен.

**Проверка устойчивости, которую стоит сделать самому.** Возьми свой корпус, посчитай
распределение `df`, посмотри, какая доля словаря имеет `df > N/2`. Обычно это доли процента
словаря — но именно эти слова встречаются в каждом втором запросе, потому что запросы пишут
на естественном языке. Частота в словаре и частота в запросах — разные вещи, и путать их
не надо.
</details>

⚠️ Ловушка D · **`sklearn.TfidfVectorizer` считает не то, что на доске.** По умолчанию у него
`smooth_idf=True` (то есть `ln((1+N)/(1+df)) + 1`), `norm='l2'` и `sublinear_tf=False`. Ни одно
из трёх решений не написано в формуле лекции, и все три меняют числа. Код при этом работает,
ошибки нет, и качество **тихо** отличается от ожидаемого. Проверим прямо сейчас — это ровно тот
тип ловушки, которому надо учить: та, что молчит.

In [ ]:
texts3 = [" ".join(["cat"] * 2 + ["dog"]), "cat dog dog mouse", "mouse cat"]
vec = TfidfVectorizer(token_pattern=r"[a-z]+")
X = vec.fit_transform(texts3)
sk_idf = dict(zip(vec.get_feature_names_out(), vec.idf_))

hand = {t: idf_classic(df, 3) for t, df in {"cat": 3, "dog": 2, "mouse": 2}.items()}
print(f"{'термин':8}{'sklearn':>10}{'формула лекции':>18}{'разница':>10}")
for t in sorted(hand):
    print(f"{t:8}{sk_idf[t]:>10.4f}{hand[t]:>18.4f}{sk_idf[t] - hand[t]:>10.4f}")
RUN["sklearn_vs_lecture_idf"] = {t: round(sk_idf[t] - hand[t], 4) for t in hand}

**Что видно.** Числа не совпали ни для одного термина, и разница не постоянная — то есть это не
«сдвиг на константу», который можно проигнорировать. Сравнивать надо не столбцы построчно,
а **знак и структуру расхождения**: `sklearn` никогда не выдаёт ноль (у него `+1` в конце),
поэтому термин `cat`, присутствующий во всех трёх документах, получает вес `1,0` вместо `0,0`.
Механизм полностью в трёх умолчаниях из ловушки выше. Чего эта таблица НЕ показывает: какая
формула «правильнее» — обе разумны, и обе используются в проде. Что делать: всегда печатать
`vectorizer.idf_` и сверять с тем, что ты думаешь, что считаешь. Молчаливое расхождение
с собственными ожиданиями — самый дорогой класс ошибок в поиске.

---

## Часть 3 · BM25 — 35 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 3.1 | Что именно делает `k1`? | считаем насыщение частоты и рисуем кривую |
| 3.2 | Что именно делает `b`? | считаем нормировку длины на реальном перекосе |
| 3.3 | Верна ли наша реализация? | сверяем все 16 значений с `data/l3-bm25.json` |

BM25 — это `idf` × насыщение(`tf`) × нормировка(длина). Три множителя, два параметра.

In [ ]:
def idf_bm25(df, n):
    return math.log((n - df + 0.5) / (df + 0.5) + 1)

def bm25_term(tf, dl, df, n, avgdl, k1=K1, b=B):
    sat = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))
    return idf_bm25(df, n) * sat

bad = []
for d in BM["docs"]:
    for t in d["terms"]:
        got = round(bm25_term(t["tf"], d["len"], t["df"], N_LEC, AVGDL_LEC), 4)
        if abs(got - t["bm25"]) > 1e-4:
            bad.append((d["id"], t["t"], got, t["bm25"]))

assert not bad, f"BM25 разошёлся с лекцией: {bad[:3]}"
n_checked = sum(len(d["terms"]) for d in BM["docs"])
print(f"сверка BM25 с data/l3-bm25.json: {n_checked} значений совпали до 1e-4")
print(f"параметры: k1={K1}, b={B}, N={N_LEC}, avgdl={AVGDL_LEC}")
RUN["bm25_lecture_check"] = n_checked

**Что видно.** Все шестнадцать покомпонентных значений BM25 совпали с лекционными до четвёртого
знака. Сравнивать надо не «похоже» с «похоже», а именно так — с допуском, объявленным заранее.
Ожидаемая картина ровно такая по механизму: `data/l3-bm25.json` порождён `_research/gen_l3.py`
той же формулой, и если бы мы ошиблись хоть в скобке, `assert` бы упал. Чего эта сверка НЕ
показывает: что BM25 хорошо ранжирует. Она показывает, что мы **правильно посчитали**, а это
разные утверждения — правильно посчитанное число может вести к неверному выводу. Что делать:
теперь, когда реализация доказана, можно спрашивать про качество — и только теперь.

<details><summary>Вывод формулы насыщения: откуда взялась именно такая дробь</summary>

Компонента насыщения в BM25 выглядит как `tf·(k1+1) / (tf + k1·(1 - b + b·dl/avgdl))`.
Дробь кажется взятой с потолка. Она не взята с потолка.

**Требования, из которых она получается.** Нужна функция `f(tf)`, у которой: `f(0) = 0`
(слова нет — вклада нет); `f` монотонно растёт; `f` имеет горизонтальную асимптоту (насыщение);
и переход к насыщению управляется одним параметром. Простейшая функция с такими свойствами —
`tf / (tf + k)`: она равна нулю в нуле, растёт, стремится к единице. Параметр `k` задаёт,
при каком `tf` достигается половина максимума: `f(k) = 0,5`.

**Откуда `(k1+1)` в числителе.** Это нормировка, чтобы при `tf = 1` и документе средней длины
вклад равнялся ровно единице: подставь `tf = 1`, `dl = avgdl`, и знаменатель станет `1 + k1`,
а числитель `1·(k1+1)` — дробь даёт единицу. Множитель нужен исключительно ради того, чтобы
шкала не зависела от `k1` и разные `k1` можно было сравнивать. На порядок ранжирования он
не влияет вообще — это константа для всех документов.

**Откуда нормировка длины внутри знаменателя.** Вместо `k1` в знаменателе стоит
`k1·(1 - b + b·dl/avgdl)`. При `b = 0` множитель равен единице — длина игнорируется. При
`b = 1` он равен `dl/avgdl` — полная нормировка. Промежуточные `b` дают частичную. Важно,
что нормировка действует **на скорость насыщения**, а не на итоговый вклад: в длинном
документе нужно больше вхождений, чтобы добраться до той же доли максимума. Это точнее, чем
просто поделить счёт на длину, — деление наказывало бы длинные документы и на первом вхождении.

**Чего этот вывод НЕ доказывает.** Он не доказывает оптимальности BM25. Он показывает, что
формула — это минимальная конструкция, удовлетворяющая четырём разумным требованиям. Другие
конструкции с теми же свойствами существуют, и некоторые из них работают лучше на конкретных
корпусах. Об этом — в следующем блоке.
</details>

### Шаг 3.1 · Что делает `k1`

Насыщение. Второе вхождение слова добавляет меньше, чем первое, десятое — почти ничего.
Соблазн очевиден: «чем чаще слово, тем релевантнее». Он неверен — документ, где `space`
встретилось сорок раз, не в сорок раз релевантнее того, где оно встретилось раз.

In [ ]:
tf_range = np.arange(0, 21)
plt.figure(figsize=(9, 3.2))
for k1 in (0.5, 1.2, 1.5, 3.0):
    sat = (tf_range * (k1 + 1)) / (tf_range + k1)
    plt.plot(tf_range, sat, label=f"k1 = {k1}")
plt.plot(tf_range, tf_range, "k--", lw=1, label="сырой tf (без насыщения)")
plt.ylim(0, 12); plt.xlabel("tf -- сколько раз слово в документе"); plt.ylabel("вклад")
plt.legend(); plt.title("Насыщение: k1 задаёт, как быстро перестаёт расти вклад")
plt.tight_layout(); plt.show()

**Что видно.** Сравнивать надо каждую сплошную кривую **с пунктиром**, а не кривые друг с другом:
пунктир — это сырой счёт, наш `BASE`, и он уходит вверх без предела. Все кривые BM25 выполаживаются,
и `k1` управляет только скоростью выхода на полку: при `k1 = 0,5` разница между `tf = 3` и `tf = 20`
почти исчезает, при `k1 = 3` она ещё заметна. Механизм — дробь с `tf` и в числителе, и
в знаменателе. Чего график НЕ показывает: какое `k1` лучше на **твоих** данных; полка — свойство
формулы, а оптимум — свойство корпуса. Что делать: не подбирать `k1` первым делом. На практике
`1,2–2,0` работает почти везде, а выигрыш от подбора обычно меньше разброса, который мы сейчас
и померяем.

<details><summary>Альтернативы BM25 и почему на курсе всё-таки BM25</summary>

BM25 — не единственная и не обязательно лучшая формула. Четыре живых конкурента:

**BM25+.** Добавляет константу `δ` к компоненте насыщения: `sat + δ`. Смысл: в очень длинных
документах нормировка длины давит вклад термина почти до нуля, и длинный релевантный документ
проигрывает короткому нерелевантному, где термина нет вовсе... точнее, где он есть один раз.
`δ ≈ 1` ставит пол под вклад присутствующего термина. На корпусах с сильным разбросом длин
даёт стабильный, хотя и небольшой выигрыш.

**BM25F.** Для документов с полями (заголовок, тело, анкоры). Наивный подход — считать BM25
по каждому полю и складывать с весами — **неверен**: насыщение применяется к каждому полю
отдельно, и десять вхождений в заголовке насыщаются раньше, чем должны. BM25F сначала
складывает взвешенные частоты по полям, и только потом применяет насыщение один раз. Разница
принципиальная и вылезает ровно там, где поля разной длины.

**Языковые модели с Дирихле-сглаживанием.** Совсем другая постановка: оценивается вероятность
породить запрос моделью документа. Сглаживание Дирихле `(tf + μ·p(t|C)) / (dl + μ)` даёт
нормировку длины «бесплатно», как следствие модели, а не как отдельный параметр `b`.
На коротких запросах сравнимо с BM25, на длинных часто лучше.

**DFR (divergence from randomness).** Семейство моделей Амати: вес термина — расхождение
между наблюдаемым распределением и случайным. Математически изящно, на практике сравнимо
с BM25 и заметно сложнее в объяснении.

**Почему BM25.** Три причины, и ни одна не «он лучший». Первая: он до сих пор умолчание
в Lucene, Elasticsearch и OpenSearch, то есть ты почти наверняка встретишь именно его.
Вторая: два параметра с ясным физическим смыслом — это лучший учебный объект, чем `μ`,
у которого смысл есть, но неинтуитивный. Третья: на нём проще всего показать, что даже
у «простой» формулы каждая скобка выведена, а не угадана.
</details>

### Шаг 3.2 · Что делает `b`

`b = 0` — длина игнорируется полностью. `b = 1` — полная нормировка. `b = 0,75` — компромисс,
который стал умолчанием, потому что работает почти везде.

In [ ]:
avgdl = float(np.mean([len(t) for t in CORPUS.values()]))
N = len(CORPUS)
df_all = {t: len(p) for t, p in post.items()}

def bm25_score(did, query, b=B, k1=K1):
    dl = doclen[did]
    return sum(bm25_term(post.get(q, {}).get(did, 0), dl, df_all.get(q, 0) or 1, N, avgdl, k1, b)
               for q in query)

def rank(query, b=B, k1=K1, top=5):
    cand = set()
    for q in query:
        cand |= set(post.get(q, {}))
    return sorted(cand, key=lambda d: (-bm25_score(d, query, b, k1), d))[:top]

for b in (0.0, 0.75, 1.0):
    top = rank(QUERY, b=b)
    print(f"b={b:<5} топ-5: {top}  длины: {[doclen[d] for d in top]}")

**Что видно.** Сравнивать надо не сами списки документов, а **длины в правой колонке**. При
`b = 0` наверх лезут документы, заметно длиннее медианы: без нормировки длинный текст набирает
больше вхождений просто потому, что он длинный. При `b = 1` картина обратная — короткие получают
преимущество, иногда чрезмерное. Механизм в знаменателе: `1 - b + b·dl/avgdl` при `b = 0`
превращается в единицу, то есть длина исчезает из формулы. Чего эта таблица НЕ показывает:
какой из трёх списков **релевантнее** — у нас нет разметки, и до неё мы дойдём на семинаре
недели 4, где будем считать nDCG. Что делать: пока не объявлять победителя. У нас нет критерия,
а выбирать победителя без критерия — это ровно то, чего мы избегаем весь курс.

<details><summary>Почему k1 и b нельзя подбирать на том же наборе, где меряешь</summary>

Соблазн очевиден: у нас есть два параметра и есть метрика, значит переберём сетку и возьмём
максимум. Это работает и даёт число, которому нельзя верить.

**Механизм.** Перебирая сетку из `n` конфигураций и выбирая максимум по метрике на тестовом
наборе, ты выбираешь не лучшую конфигурацию, а конфигурацию, которой больше всех повезло
на этом конкретном наборе. Ожидаемое смещение растёт с числом конфигураций и падает
с размером набора. На двадцати запросах и сетке 10×10 смещение легко превышает всю разницу
между разумными значениями `b`.

**Как правильно.** Три набора запросов: на первом подбираешь, на втором проверяешь, что
подбор не переобучился, на третьем — финальное число, которое ты называешь вслух. Если
запросов мало (а их всегда мало), кросс-валидация по запросам: делишь запросы на пять групп,
подбираешь на четырёх, меряешь на пятой, повторяешь пять раз. Число, которое ты публикуешь, —
среднее по пяти замерам, и рядом с ним обязательно разброс.

**Чего это НЕ означает.** Не означает, что подбирать бесполезно. На корпусе, непохожем
на TREC-новости, разница между `b = 0,3` и `b = 0,75` может быть большой и настоящей.
Означает только, что «мы подобрали и получили +2%» без отложенного набора — это не результат,
а описание процедуры.

**Наш случай.** Мы вообще не подбирали, потому что у нас нет разметки. Это честнее, чем
подобрать по метрике, которой нет. Разметка появится на следующем занятии.
</details>

⚠️ Ловушка C · **Выше скор — не значит релевантнее.** Мы только что видели, как меняется топ
от параметра. Ни одна из трёх выдач не «правильная»: правильность определяется разметкой,
а её здесь нет. Число посчитано верно, вывод «`b=0.75` лучше» из него не следует.

⚠️ Ловушка F · **Чужое `b` — не твоё `b`.** Умолчание `0,75` подобрано на TREC-коллекциях
из новостных документов. На корпусе, где длины однородны, `b` почти не влияет; на корпусе
из title-подобных строк оптимум уезжает к нулю. Переносить `0,75` как «правильное значение»
нельзя — переносится только рассуждение, зачем этот параметр вообще нужен.

### Шаг 3.3 · Шум раньше эффекта

Прежде чем утверждать, что `b = 0,75` лучше `b = 0,7`, надо узнать, **больше ли разница, чем
разброс**. Разброса между сидами у нас нет — BM25 детерминирован. Зато есть разброс между
**подвыборками корпуса**, и он играет ту же роль.

In [ ]:
def overlap_at5(b1, b2, sample_ids):
    sub = {d: CORPUS[d] for d in sample_ids}
    p2, dl2 = build_index(sub)
    n2 = len(sub); a2 = float(np.mean(list(dl2.values())))
    df2 = {t: len(p) for t, p in p2.items()}

    def rk(b):
        cand = set()
        for q in QUERY:
            cand |= set(p2.get(q, {}))
        sc = {d: sum(bm25_term(p2.get(q, {}).get(d, 0), dl2[d], df2.get(q, 0) or 1, n2, a2, K1, b)
                     for q in QUERY) for d in cand}
        return sorted(cand, key=lambda d: (-sc[d], d))[:5]
    return len(set(rk(b1)) & set(rk(b2))) / 5

ids = list(CORPUS)
rng = np.random.default_rng(SEED)
diff_b = []
for _ in range(8):
    half = list(rng.choice(ids, size=len(ids) // 2, replace=False))
    diff_b.append(overlap_at5(0.70, 0.80, half))

print(f"пересечение топ-5 при b=0.70 против b=0.80 на одной подвыборке: "
      f"{np.mean(diff_b):.2f} (разброс {np.min(diff_b):.2f}--{np.max(diff_b):.2f})")
RUN["overlap_b070_b080"] = float(np.mean(diff_b))

**Что видно.** Пересечение топ-5 при `b = 0,70` и `b = 0,80` близко к единице, и разброс между
подвыборками сопоставим с самой разницей. Сравнивать надо среднее **с шириной интервала**, а не
среднее с единицей: если разброс перекрывает эффект, по одному прогону сказать «`0,80` лучше»
нельзя — что бы ни напечатала ячейка выше. Ожидаемая картина такая по механизму: `b` в этом
диапазоне двигает знаменатель на проценты, а порядок документов меняется скачками. Чего этот
замер НЕ показывает: поведения на краях — между `b = 0` и `b = 1` разница огромна, мы её видели.
Что делать: считать, что подбор `b` в третьем знаке — это подгонка под шум. Экономит день.

<details><summary>Почему мы мерили пересечение, а не метрику качества, и что теряется</summary>

Пересечение топ-5 — это мера **стабильности**, а не качества. Мы выбрали её осознанно
и заплатили за это.

**Что она ловит.** Если два конфига дают одинаковые списки, между ними невозможно выбрать
по качеству — они эквивалентны для пользователя. Если пересечение мало, разница есть,
и вопрос «какая лучше» становится осмысленным. То есть пересечение отвечает на вопрос
«есть ли вообще что обсуждать», и отвечает **без разметки**. Это дёшево и это ставит границу.

**Что она теряет.** Всё про правильность. Два конфига могут давать одинаково плохие списки
с пересечением 1,0. Могут давать пересечение 0,2, где один список отличный, а другой мусорный.
Пересечение симметрично, а качество — нет.

**Более честные бесплатные меры.** Ранговая корреляция Кендалла по всему списку кандидатов,
а не по топ-5: она учитывает, насколько сильно переставились документы, а не только факт
перестановки. И RBO (rank-biased overlap) — то же, но с весами, убывающими по рангу, что
ближе к тому, как смотрит пользователь. Обе считаются без разметки.

<summary>Как сделать правильно, если есть бюджет</summary>
Взять двадцать запросов вместо одного и показать распределение пересечений, а не одно число.
Один запрос — это одна точка, а по одной точке про эффект не говорят. Мы взяли один запрос
ради времени, и это ограничение нашего замера, а не свойство метода.
</details>

⚠️ Ловушка D · **`np.random.choice` по списку строк тихо медленный и тихо меняет порядок.**
Мы фиксируем генератор через `default_rng(SEED)`, иначе два запуска ноутбука дадут разные
подвыборки и разные числа — и ты будешь искать ошибку в формуле там, где её нет.

⚠️ Ловушка A · **Половинная подвыборка меняет `N` и `avgdl`.** Мы честно пересчитываем индекс
внутри `overlap_at5`, а не переиспользуем глобальные `N` и `avgdl`. Если этого не сделать, `idf`
считается по одному корпусу, а частоты — по другому, и результат выглядит правдоподобно, оставаясь
бессмысленным. Это ровно тот тип ошибки, который не падает.

### Шаг 3.4 · Замер без модели

**Замер без модели** — потолок задачи, который не зависит ни от какой формулы. Считаем
лексическое пересечение запроса с документами: если слова запроса физически не встречаются
в релевантном документе, **никакой** лексический ранжировщик его не найдёт. Это граница сверху
для всего сегодняшнего занятия и половина мотивации всего дальнейшего курса.

In [ ]:
vocab_docs = {d: set(t) for d, t in CORPUS.items()}
reachable = sum(1 for s in vocab_docs.values() if any(q in s for q in QUERY))
both = sum(1 for s in vocab_docs.values() if all(q in s for q in QUERY))
print(f"документов всего: {len(CORPUS)}")
print(f"содержат хотя бы одно слово запроса: {reachable} ({reachable / len(CORPUS):.1%})")
print(f"содержат оба слова:                  {both} ({both / len(CORPUS):.1%})")
print(f"НЕДОСТИЖИМЫ лексически:              {len(CORPUS) - reachable} "
      f"({1 - reachable / len(CORPUS):.1%})")
RUN["lexical_ceiling"] = reachable / len(CORPUS)

**Что видно.** Значительная доля корпуса лексически недостижима этим запросом — там нет ни слова
`space`, ни слова `team`. Сравнивать надо не BM25 с TF-IDF, а **любой из них с этой границей**:
сколько ни улучшай формулу, эти документы она не вернёт никогда. Механизм тривиален: лексический
поиск ищет совпадения строк, а не смыслов; документ про `orbital mission` про космос, но слова
`space` в нём может не быть. Чего этот замер НЕ показывает: сколько из недостижимых на самом деле
релевантны — для этого нужна разметка. Что делать: запомнить число. Это и есть мотивация плотного
поиска из L6 и всей второй половины курса: не «нейросети моднее», а «у лексики есть потолок,
и вот он в процентах».

<details><summary>Ограничения этого семинара, которые надо назвать вслух</summary>

Полный список того, где мы срезали угол, и в какую сторону это смещает выводы.

**Один запрос.** Всё занятие мы работали с `["space", "team"]`. Один запрос — это одна точка,
и любое утверждение вида «`b` не влияет» верно ровно для неё. Смещение: в пользу вывода
«разницы нет», потому что на конъюнкции двух частых терминов кандидатов много и топ устойчив.
На редком однословном запросе картина была бы другой.

**Нет разметки.** Мы ни разу не сказали «лучше» — и это честно, но это и означает, что
половина инженерной работы сегодня не сделана. Метрики появятся на следующем занятии,
и только там сравнения станут содержательными.

**Восемь подвыборок.** Разброс оценён по восьми, а не по сотне. Восемь — это минимум,
на котором интервал вообще имеет смысл, и он широкий. Смещение: в пользу вывода «эффект
в шуме», потому что при малом числе повторов шум переоценивается.

**Плохой baseline в замере скорости.** Сказано в отдельном блоке выше: наш скан заведомо
неэффективен. Смещение — в пользу индекса, и величину смещения мы не измерили.

**Корпус игрушечный по меркам поиска.** Две тысячи документов — это меньше, чем в любом
проде на три порядка. Всё, что мы видели про постинги и `avgdl`, верно качественно;
количественно на миллионах документов доминируют совсем другие вещи — сжатие постингов,
локальность обращений к диску, кэш.

**Что из этого можно исправить прямо сейчас.** Первое и третье — просто больше вычислений,
и ноутбук их выдержит: поставь `SMOKE=0`, добавь запросов и повторов. Второе исправить нельзя
без разметки. Четвёртое исправляется десятью строками. Пятое не исправляется в рамках
двухчасового занятия вообще, и это нормально: семинар воспроизводит **явление**, а не масштаб.
</details>

<details><summary>Лексический потолок: как он связан с recall и почему это мотивация всего курса</summary>

Число, которое мы посчитали, — верхняя граница полноты для **любого** метода, работающего
на совпадении строк. Ни BM25, ни BM25F, ни идеально подобранные `k1` и `b` не поднимут её
ни на процент. Это редкий случай, когда граница считается за три строки кода и не зависит
ни от какой модели.

**Связь с recall.** Если доля лексически достижимых документов равна `r`, то `Recall@∞`
любого лексического ретривера не превосходит `r`, взятого по **релевантным** документам.
Мы посчитали `r` по всему корпусу, а не по релевантным, — это приближение, и оно может
быть как оптимистичным, так и пессимистичным. Честная оценка требует разметки.

**Три способа поднять потолок, и все три — темы следующих лекций.**
Первый: расширение запроса — синонимы, морфология, переписывание. Дёшево, ломает точность,
и мы разбираем это в задании 3 и подробно в L17.
Второй: расширение документа — doc2query, где модель генерирует запросы, на которые документ
отвечает, и они дописываются в индекс. Потолок поднимается, индекс растёт.
Третий: отказ от совпадения строк — плотные векторы, где близость меряется в пространстве
смыслов. Это L6 и весь дальнейший курс, и именно поэтому он существует.

**Чего это НЕ означает.** Не означает, что лексический поиск устарел. У него есть свойство,
которого нет у плотного: он **никогда не промахивается по точному совпадению**. Запрос
с артикулом товара, номером ошибки или фамилией лексический поиск находит всегда, а плотный
может не найти. Отсюда гибриды из L12: не потому, что «два лучше одного», а потому что
у методов разные типы промахов.
</details>

---

## Задания — 25 мин

Ты пишешь мало кода — это не «заполни пропуски». Ты читаешь, запускаешь, объясняешь и применяешь.
Каждое задание заканчивается `assert`-самопроверкой.

**Про самопроверку честно:** пройденная самопроверка не гарантирует, что задание сделано
осмысленно, но проваленная гарантирует, что где-то ошибка.

### Задание 1 · Сглаженный `idf` и его граница

**Что сделать.** Реализуй `idf_rsj(df, n) = ln((n - df + 0.5) / (df + 0.5))` — несглаженный
вариант Робертсона-Спарк Джонс, без `+1` под логарифмом, — и **честно сравни** его с нашим
`idf_bm25` на всём диапазоне `df` от 1 до `N`.

**Что нужно получить.** `neg_from` — минимальное `df`, при котором `idf_rsj` становится
отрицательным. Тип: `int`.

**Подсказка.** `next(df for df in range(1, N + 1) if idf_rsj(df, N) < 0)`.

**Прочитай до запуска.** Ожидаемых исходов три, и все содержательны:
* `neg_from ≈ N/2` — так и должно быть: при `df > N/2` числитель меньше знаменателя;
* `neg_from` не нашёлся — ты потерял `-df` в числителе, проверь скобки;
* `neg_from = 1` — перепутал местами числитель и знаменатель.

**Формулировка вывода.** Не «RSJ хуже», а: **при каких условиях** отрицательный вес термина
становится проблемой. Подумай, что произойдёт с документом, который содержит слово из запроса,
если вес этого слова отрицательный.

In [ ]:
# --- твой код: ЗАДАНИЕ 1 ---
def idf_rsj(df, n):
    ...

neg_from = ...
# --- конец ---

assert isinstance(neg_from, int), "neg_from должен быть int -- минимальное df, а не значение idf"
assert idf_rsj(neg_from, N_LEC) < 0 <= idf_rsj(neg_from - 1, N_LEC), \
    "neg_from не является ГРАНИЦЕЙ: слева от него idf обязан быть неотрицательным"
assert idf_bm25(neg_from, N_LEC) > 0, \
    "сглаженный idf не должен уходить в минус НИ ПРИ КАКОМ df -- в этом весь смысл +1"
print(f"RSJ уходит в минус начиная с df = {neg_from} из N = {N_LEC}")

### Задание 2 · Фальсифицируемый тезис про `b`

**Тезис.** *Нормировка длины помогает тем сильнее, чем разнороднее длины документов.*
Если это так, то на подвыборке с **однородными** длинами топ-5 при `b = 0` и `b = 1` совпадёт
сильнее, чем на подвыборке с разнородными.

**Что сделать.** Собери две подвыборки по 300 документов: `homog` — документы с длиной в узком
коридоре вокруг медианы, `heterog` — половина самых коротких плюс половина самых длинных.
Посчитай для каждой пересечение топ-5 при `b = 0` и `b = 1` и **честно сравни**.

**Что нужно получить.** `ov_homog`, `ov_heterog` — доли пересечения, `float` в `[0, 1]`.

**Подсказка.** `overlap_at5(0.0, 1.0, ids_подвыборки)` уже написан выше.

**Прочитай до запуска.** Все исходы содержательны:
* `ov_homog > ov_heterog` — тезис подтвердился, нормировка важнее там, где длины пляшут;
* примерно равны — эффект меньше разброса, и это **тоже результат**: он говорит, что на этом
  корпусе `b` можно не трогать, и экономит тебе день подбора;
* `ov_homog < ov_heterog` — тезис опровергнут; посмотри, действительно ли `homog` однороден,
  и не собрал ли ты его из двух пиков.

**Формулировка вывода.** Не про то, какое число получилось, а про то, **при каком разбросе длин
разница вообще измерима** на выборке твоего размера.

In [ ]:
lens_arr = np.array([doclen[d] for d in ids])
med = np.median(lens_arr)

# --- твой код: ЗАДАНИЕ 2 ---
homog = ...     # список id с длиной близко к медиане, ровно 300 штук
heterog = ...   # 150 самых коротких + 150 самых длинных
# --- конец ---

ov_homog = overlap_at5(0.0, 1.0, homog)
ov_heterog = overlap_at5(0.0, 1.0, heterog)

assert len(homog) == len(heterog) == 300, \
    "подвыборки должны быть ОДНОГО размера -- иначе меняешь два фактора сразу"
assert np.std([doclen[d] for d in homog]) < np.std([doclen[d] for d in heterog]), \
    "homog обязан быть однороднее heterog по построению -- иначе сравнение бессмысленно"
print(f"однородные: пересечение {ov_homog:.2f} · разнородные: {ov_heterog:.2f}")
RUN["task2"] = {"ov_homog": ov_homog, "ov_heterog": ov_heterog}

### Задание 3 · Словами, а не числом

**Что сделать.** В ячейке markdown ниже ответь **словами** на вопрос: в шаге 3.4 мы посчитали,
что часть корпуса лексически недостижима запросом. Кто-то предлагает починить это, добавив
в запрос синонимы (`space` → `space OR orbital OR cosmos`).

Назови **два** способа, которыми это ухудшит поиск, и для каждого скажи, каким замером ты бы
это поймал. Один из двух должен быть про метрику, а не про качество выдачи.

**Прочитай до запуска.** Здесь нет `assert` на текст — его проверяет человек. Но самопроверка
ниже ловит главное: что ты не оставил ячейку пустой и ответил про **оба** требуемых аспекта.

**Формулировка вывода.** Не «синонимы — это плохо», а: при каком условии расширение запроса
окупается, и как ты узнаешь, что это условие выполнено.

In [ ]:
# --- твой код: ЗАДАНИЕ 3 ---
ANSWER_3 = """
Впиши ответ сюда: минимум 60 слов, два способа ухудшения и по замеру на каждый.
"""
# --- конец ---

assert len(ANSWER_3.split()) >= 60, "ответ короче 60 слов -- два способа с замерами так не уместить"
assert "ЗАДАНИЕ 3" not in ANSWER_3, "заглушка не заменена"
print(f"ответ принят: {len(ANSWER_3.split())} слов")

---

## Итог занятия — 5 мин

Что мы сделали и, главное, чего **не** делали.

* Построили инвертированный индекс и замерили его выигрыш **отношением**, а не секундами.
* Воспроизвели `idf` и все 16 значений BM25 из `data/l3-bm25.json` — реализация доказана,
  а не «выглядит правдоподобно».
* Показали, что `sklearn` считает другой `idf`, и это молчаливое расхождение.
* Померяли разброс **до** утверждений об эффекте `b` — и отказались объявлять победителя
  без разметки.
* Посчитали лексический потолок: сколько документов не найдёт **никакой** лексический метод.

**Ограничение нашего замера, которое надо назвать вслух.** У нас не было ни одной оценки
релевантности. Всё, что мы сравнивали, — это порядок и пересечение, а не качество. Поэтому
сегодня не прозвучало ни одного «лучше». Разметка появится на следующем занятии, вместе
с nDCG, — и только там слово «лучше» станет законным.

**Что мы будем и чего не будем замерять дальше.** Абсолютные секунды из этого ноутбука
непереносимы: бесплатный Colab делит процессор. Переносимы отношения и качественные выводы.
Сравнивать твои числа с числами соседа можно только внутри одной конфигурации `RUN`.

<details><summary>Шесть типов ловушек, которые сегодня встретились — и почему учат именно тихим</summary>

За занятие мы прошли все шесть типов из стандарта курса. Собери их вместе — это и есть
карта того, как поиск ломается на практике.

**A · данных.** Заголовки 20NG с именем категории прямо во входе; восемь самых коротких
документов на слайде как «искусственный масштаб»; половинная подвыборка, меняющая `N`
и `avgdl`. Общее свойство: данные не те, что ты думаешь, а код об этом молчит.

**B · метрики.** Ускорение поиска в сто раз, дающее два процента ускорения продукта.
Метрика посчитана верно и меряет не то, что тебя волнует.

**C · интерпретации.** Совпадение порядка на трёх документах как «доказательство»; более
высокий скор как «релевантнее». Число верное, вывод — нет.

**D · инструмента.** `TfidfVectorizer` с тремя умолчаниями, ни одно из которых не написано
на доске; `np.random.choice` без фиксированного генератора. Код работает, ошибки нет,
результат тихо другой.

**E · замера.** Отсутствие прогрева и усреднение вместе с первым прогоном. Бенчмарк испорчен
методикой, а не кодом.

**F · переноса.** `b = 0,75` как «правильное значение»; чужие тайминги на чужом железе.
Число настоящее, но не твоё.

**Почему именно эти.** Ловушка, которая падает с трейсбеком, семинара не требует: ты увидишь
её сам через десять секунд. Учить надо тем, которые **молчат** — и все шесть выше молчат.
Ни одна из них не выбросит исключение, ни одна не окрасит ячейку красным. Каждая даст
правдоподобное число, которое ты запишешь в отчёт.

Отсюда единственное правило, которое дороже остальных: **сомневайся в собственном числе
раньше, чем в чужом**. Чужое ты и так проверишь. Своё — то, которое напечатала твоя ячейка
и которое совпало с ожиданием, — проверять не хочется совсем, и именно оно чаще всего неверно.
</details>

In [ ]:
import hashlib
import sklearn

# Версии -- без них через полгода файл будет утверждать, что числа те же, а они поедут.
RUN["versions"] = {"numpy": np.__version__, "sklearn": sklearn.__version__,
                   "python": ".".join(map(str, __import__("sys").version_info[:3]))}
# Хеш корпуса -- корпус может смениться под тем же именем.
RUN["corpus_sha1"] = hashlib.sha1(
    "\n".join(f"{d}:{doclen[d]}" for d in sorted(CORPUS)).encode()).hexdigest()[:12]

# Выдачи, а не только метрики: позволяют пересчитать ЛЮБУЮ метрику задним числом,
# не перезапуская индекс. Весят килобайты, экономят часы.
RUNS_OUT = {f"b={b}": rank(QUERY, b=b, top=100) for b in (0.0, 0.75, 1.0)}
RUNS_OUT["base_raw_count"] = order_base

RUN["finished"] = True
out = ARTIFACTS / "run.json"
out.write_text(json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")

runs_path = ARTIFACTS / "runs.json"
runs_path.write_text(json.dumps(RUNS_OUT, ensure_ascii=False), encoding="utf-8")

index_out = ARTIFACTS / "bm25_index.json"
index_out.write_text(json.dumps(
    {"doclen": doclen, "df": df_all, "avgdl": avgdl, "N": N, "k1": K1, "b": B,
     "postings": {t: p for t, p in list(post.items())[:5000]}},
    ensure_ascii=False), encoding="utf-8")

print(f"замеры -> {out} ({len(RUN)} ключей, версии и sha1 корпуса внутри)")
print(f"выдачи -> {runs_path} ({len(RUNS_OUT)} конфигураций по 100 документов)")
print(f"индекс -> {index_out} ({index_out.stat().st_size / 1e6:.1f} МБ)")
print(f"корпус sha1: {RUN['corpus_sha1']} · версии: {RUN['versions']}")
print("на неделе 7 lab-cascade подхватит этот индекс как первую ступень")

**Что видно.** Три артефакта на диске, и в `run.json` рядом с числами лежат версии библиотек
и sha1 корпуса. Сравнивать надо не размеры файлов, а **что из этого нельзя восстановить**:
индекс пересчитывается за секунды, а вот условия прогона — нет, их надо записать в момент
прогона или потерять навсегда. Механизм важен: через полгода `sklearn` обновится, числа поедут,
и без записанной версии файл будет молча утверждать, что всё то же самое. Отдельно обрати
внимание на `runs.json`: там лежат сами **выдачи**, по сто документов на конфигурацию. Они весят
килобайты и позволяют пересчитать любую метрику задним числом, не запуская индекс, — на неделе 4
мы этим и воспользуемся. Чего этот вывод НЕ показывает: совместимости с ноутбуком недели 7 —
её проверит уже он. Что делать: не удалять папку `artifacts`.

<details><summary>Что с этим артефактом сделает семинар недели 7 — и почему формат именно такой</summary>

Мы сохранили постинги, длины документов, `df`, `N` и `avgdl`. Это не произвольный набор:
это ровно то, что нужно, чтобы посчитать BM25 для **любого** запроса, не трогая исходные
тексты. Тексты весят на порядок больше и на следующем занятии не понадобятся.

**Как это будет использовано.** На неделе 7 (`lab-cascade`) BM25 станет первой ступенью
двухступенчатого каскада: он отберёт сотню кандидатов из корпуса, а кросс-энкодер переранжирует
эту сотню. Ключевая мысль того занятия — что качество каскада ограничено сверху полнотой
первой ступени: то, чего BM25 не вернул в сотню, кросс-энкодер уже не спасёт. Твой сегодняшний
лексический потолок — прямой предок того ограничения.

**Почему JSON, а не pickle.** Pickle быстрее и компактнее, и он же привязан к версии Python
и к твоим классам. Через полгода он не откроется, а JSON откроется чем угодно. Для артефакта,
который живёт между занятиями и который студент может открыть глазами, читаемость важнее
пары мегабайт. Для миллиона документов ответ был бы другим — там нужен бинарный формат
со сжатием постингов.

**Обрезка на пяти тысячах терминов.** В коде стоит `list(post.items())[:5000]` — мы пишем
не весь словарь. Это сознательная экономия, и она **некорректна** для честного индекса:
выброшенные термины просто не найдутся. Для учебного артефакта, где запросы известны заранее,
это приемлемо; в проде — нет. Ограничение названо, а не спрятано, и если тебе нужен полный
индекс, убери срез: файл вырастет примерно втрое.

**Почему каждый семинар всё равно запускается один.** Если ты пропустил это занятие, ноутбук
недели 7 не упадёт: он проверит наличие файла, не найдёт и построит свой индекс на уменьшенной
выборке, сказав об этом вслух. Накопительная система — удобство, а не обязательство. Пропуск
занятия стоит времени пересчёта, но не выкидывает тебя из курса.
</details>

<details><summary>Что именно надо класть в run.json, чтобы через месяц себе поверить</summary>

Правило простое: в файл с числами кладётся всё, без чего число нельзя воспроизвести.
Минимальный список, который мы выполнили частично:

* **Конфигурация прогона** — `SEED`, размеры выборок, все параметры формул. У нас есть.
* **Версии.** Записаны: `numpy`, `sklearn` и сам Python. Через полгода библиотека обновится,
  числа поедут, и только эта строка позволит понять, почему.
* **Хеш данных.** Записан: `sha1` от отсортированного списка «документ:длина». Стоит
  миллисекунды и снимает целый класс споров «а тот же ли это корпус».
* **Время и железо.** Не ради сравнения — ради понимания, почему на перезапуске цифры другие.
* **Отрицательные результаты.** Мы записали `overlap_b070_b080`, хотя эффекта не нашли.
  Это важнее положительных: именно отсутствие эффекта чаще всего забывают записать,
  а потом переоткрывают заново.

**Выдачи сохранены** — `runs.json`, по сто документов на каждую конфигурацию `b` плюс базовый
сырой счёт. Это тот самый дешёвый приём: килобайты на диске, а взамен возможность пересчитать
nDCG, MAP, RBO или что угодно ещё, не притрагиваясь к индексу.

**Что осталось незаписанным.** Время и железо — не ради сравнения, а чтобы понимать, почему
на перезапуске цифры другие. И сами **скоры**, а не только порядок: имея скоры, можно строить
гибридное слияние, не пересчитывая ступени. Второе аукнется на неделе 7, и там же будет закрыто.
</details>

---

## Решения

**Подглядеть — не поражение. Поражение — уйти с занятия, не поняв, где был затык.**

<details><summary>Задание 1 · сглаженный idf и его граница</summary>

```python
def idf_rsj(df, n):
    return math.log((n - df + 0.5) / (df + 0.5))

neg_from = next(df for df in range(1, N_LEC + 1) if idf_rsj(df, N_LEC) < 0)
```

Граница ровно там, где числитель сравнивается со знаменателем: `n - df + 0.5 = df + 0.5`,
то есть `df = n/2`. Смысл отрицательного веса: документ, содержащий слово из запроса, получает
**штраф** за него. На запросе из двух слов, одно из которых очень частое, документ с обоими
словами может проиграть документу с одним — что абсурдно для пользователя и совершенно
логично для формулы. `+1` под логарифмом в BM25 существует ровно затем, чтобы этого не было.
</details>

<details><summary>Задание 2 · фальсифицируемый тезис про b</summary>

```python
order = sorted(ids, key=lambda d: doclen[d])
mid = len(order) // 2
homog = order[mid - 150: mid + 150]
heterog = order[:150] + order[-150:]
```

Типичный исход: `ov_homog` заметно выше `ov_heterog` — на однородных длинах `b` почти ничего
не меняет, потому что `dl/avgdl ≈ 1` у всех и знаменатель постоянен. Но бывает и второй исход:
разница попадает в разброс. Если у тебя так — это не провал задания. Это означает, что на
подвыборке в 300 документов эффект не измерим, и правильный вывод звучит как «нужна выборка
больше или запрос с большим числом кандидатов», а не «`b` не работает».
</details>

<details><summary>Задание 3 · про синонимы</summary>

Два способа из многих:

1. **Точность падает быстрее, чем растёт полнота.** `cosmos` вытащит документы про косметику
   и про сериал, и в топ-3 — а пользователь видит именно топ-3 — станет хуже, даже если
   Recall@100 вырос. Замер: precision@3 до и после, на одной разметке.
2. **Метрика начинает мерить не то.** Если расширение делается **и** в индексе, и в запросе,
   `df` синонимов меняется, а с ним `idf` — то есть меняется шкала, в которой ты сравниваешь
   «до» и «после». Замер: сравнить `idf` общих терминов до и после расширения; если они уехали,
   твои две выдачи посчитаны разными формулами, и сравнивать их нельзя.

Условие окупаемости: расширение окупается, когда лексический потолок из шага 3.4 действительно
режет **релевантные** документы, а не просто снижает их ранг. Проверяется это разметкой,
а не интуицией.
</details>

---

## Литература

Всё, на что опирается это занятие. Первые две ссылки читаются за вечер и стоят того.

* **Robertson & Zaragoza (2009), «The Probabilistic Relevance Framework: BM25 and Beyond»** —
  источник формулы, которую мы сегодня считали. Разделы 3 и 4 — это ровно вывод из нашего
  блока про насыщение, только подробнее и с историей. Там же честно сказано, какие поправки
  инженерные, а какие следуют из модели.
* **Manning, Raghavan & Schütze, «Introduction to Information Retrieval»**, главы 1, 2 и 6 —
  инвертированный индекс, слияние постингов, skip-pointers и векторная модель. Книга бесплатна
  на сайте Стэнфорда.
* **Trotman, Puurula & Burgess (2014), «Improvements to BM25 and Language Models Examined»** —
  сравнение BM25, BM25+ и языковых моделей на одинаковых условиях. Полезна тем, что показывает,
  насколько небольшими бывают разницы, которые в отдельных статьях подаются как прорыв.
* **Документация `sklearn.feature_extraction.text.TfidfVectorizer`** — раздел про то, как
  именно считается `idf`. Три абзаца, которые сегодня спасли бы полдня.
* **Лекция L3 «Классический ИП»** и `data/l3-*.json` — числа, с которыми мы сверялись.
  Файлы порождены `_research/gen_l3.py`; если хочешь другую выборку документов, менять надо
  генератор, а не JSON.

**Дальше по курсу.** L5 даст метрики, и слово «лучше» станет законным. L6 объяснит, чем
заменяют совпадение строк, когда упираешься в лексический потолок. L12 покажет, зачем
лексический и плотный поиск соединяют, вместо того чтобы выбирать между ними.